In [2]:
import os
import sys
import pandas as pd
from ydata_profiling import ProfileReport
current = os.getcwd()
path_to_root = os.path.join (current, '../')
abs_path = os.path.abspath(path_to_root)
sys.path.append(abs_path)
import config
import ijson
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

client = config.create_minio_client()

/home/panos-varitis/anaconda3/envs/thesis/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[Bucket('aws'), Bucket('azure'), Bucket('azure-clean'), Bucket('google'), Bucket('google-clean')]


In [15]:
# object_name = "AmazonEC2.json"
object_name = "AmazonTimestream.json"

TARGET_RECORDS = 3000 
sample_products = []

response = client.get_object(config.PROVIDERS.get("aws").get("bucket"), object_name=object_name)

try:
    # Το ijson διαβάζει κατευθείαν από το stream byte-byte
    parser = ijson.kvitems(response, 'products')
    
    count = 0
    for sku, product_data in parser:
        #Mε την προοπτική να δημιουργεί πεδίο με sku με την αντίστοιχη τιμή μέσα στο dict αλλά αυτό ήδη υπάρχει
        # product_data['sku'] = sku
        sample_products.append(product_data)
        
        count += 1
        if count >= TARGET_RECORDS:
            break
            
    print(f"Downloaded  {len(sample_products)} records μέσω streaming.")

finally:
    response.close()
    response.release_conn()

# Μετατροπή σε αρχικό DataFrame
df_aws = pd.json_normalize(sample_products)
print(f"DataFrame: Rows = {df_aws.shape[0]}, Columns = {df_aws.shape[1]}")
df_aws.head()

Downloaded  973 records μέσω streaming.
DataFrame: Rows = 973, Columns = 25


,sku,productFamily,attributes.servicecode,attributes.location,attributes.locationType,attributes.instanceType,attributes.vcpu,attributes.memory,attributes.storage,attributes.networkPerformance,attributes.engineCode,attributes.databaseEngine,attributes.licenseModel,attributes.deploymentOption,attributes.usagetype,attributes.operation,attributes.normalizationSizeFactor,attributes.regionCode,attributes.servicename,attributes.description,attributes.storageMedia,attributes.volumeType,attributes.minVolumeSize,attributes.maxVolumeSize,attributes.disableactivationconfirmationemail
0,9M9RM284G374U245,Database Instance,AmazonTimestream,Asia Pacific (Mumbai),AWS Region,db.influx.24xlarge,96,768 GiB,InfluxDb Compatible,Up to 40 Gigabit,1,InfluxDB,No license required,Multi-AZ,APS3-MultiAZUsage-Db.influx.24xlarge,Compute:001,384,ap-south-1,Amazon Timestream,NaN,NaN,NaN,NaN,NaN,NaN
1,9R7TG9RQ5D8QJSZ5,Database Instance,AmazonTimestream,Asia Pacific (Sydney),AWS Region,db.influxIOIncluded.24xlarge,96,768 GiB,InfluxDb Compatible,Up to 40 Gigabit,2,InfluxDB,No license required,Cluster,APS2-ClusterNodeUsage-Db.influxIOIncluded.24xl...,Compute:002,192,ap-southeast-2,Amazon Timestream,NaN,NaN,NaN,NaN,NaN,NaN
2,ZXQNH69MUXTTEUCD,Influx Optimized Storage,AmazonTimestream,US East (N. Virginia),AWS Region,NaN,NaN,NaN,NaN,NaN,1,InfluxDB,NaN,Single-AZ,USE1-SingleAZ-InfluxDBStorage-InfluxIOIncludedT3,Storage:001,NaN,us-east-1,Amazon Timestream,SingleAZ Influx IOPS Included (16K IOPS),SSD,Influx IOPS Included with 16K IOPS,400 GB,16 TB,NaN
3,Z9UMVX66CBK745V7,Database Instance,AmazonTimestream,Asia Pacific (Sydney),AWS Region,db.influxIOIncluded.12xlarge,48,384 GiB,InfluxDb Compatible,Up to 20 Gigabit,2,InfluxDB,No license required,Cluster,APS2-ClusterNodeUsage-Db.influxIOIncluded.12xl...,Compute:002,96,ap-southeast-2,Amazon Timestream,NaN,NaN,NaN,NaN,NaN,NaN
4,A66WQEVBSFUP6U3T,Database Instance,AmazonTimestream,EU (Milan),AWS Region,db.influx.24xlarge,96,768 GiB,InfluxDb Compatible,Up to 40 Gigabit,1,InfluxDB,No license required,Cluster,EUS1-ClusterNodeUsage-Db.influx.24xlarge,Compute:001,192,eu-south-1,Amazon Timestream,NaN,NaN,NaN,NaN,NaN,NaN


In [16]:
# Στο πάνω κελί είχα μία λίστα η οποία είχε μέσα n sku, μαζί με όλα τα attributes τουσ.
# Τώρα στο βήμα αυτό κάνω access την λίστα και απομονώνω σε ένα set μόνο τον κωδικό των n sku (η επιλογή set βασίζεται στην γρήγορη αναζήτηση)
target_skus = {p['sku'] for p in sample_products}

# Αυτή θα είναι η αντίστοιχη sample products του πάνω βήματος. Θα κρατάει τα ζευγάρια sku, με στοιχεία πληρωμής, και μετά θα την κάνουμε dataframe
terms_list = []

parser = client.get_object(config.PROVIDERS.get("aws").get("bucket"), object_name=object_name)

try:
    # Πάμε βαθύτερα και από το λεξικό terms θα στοχεύσουμε μόνο στις On-Demand υπηρεσίες.
    terms_parser = ijson.kvitems(parser, 'terms.OnDemand')

    # Προφανώς δεν τα θέλω όλα!!Μόνο εκείνα των οποίων το sku βρίσκεται στο set το οποίο δημιούργησα
    for sku, term_offers in terms_parser:
        if sku not in target_skus:
            continue
        
        # Αποθηκεύουμε το sku και ολόκληρο το raw λεξικό των terms του
        terms_list.append({
            'sku': sku,
            'terms_ondemand_raw': term_offers
        })
        
        # Aπλά για να βεβαιωθώ ότι όσα products πήρα άλλες τόσες και οι τιμές 
        if len(terms_list) >= len(target_skus):
            break
            
    print(f"Downloaded {len(terms_list)} matching terms μέσω streaming.")

finally:
    parser.close()
    parser.release_conn()


df_terms = pd.DataFrame(terms_list)
print(f"Terms DataFrame: Rows = {df_terms.shape[0]}, Columns = {df_terms.shape[1]}")
df_terms.head()

Downloaded 973 matching terms μέσω streaming.
Terms DataFrame: Rows = 973, Columns = 2


,sku,terms_ondemand_raw
0,9M9RM284G374U245,{'9M9RM284G374U245.JRTCKXETXF': {'offerTermCod...
1,9R7TG9RQ5D8QJSZ5,{'9R7TG9RQ5D8QJSZ5.JRTCKXETXF': {'offerTermCod...
2,ZXQNH69MUXTTEUCD,{'ZXQNH69MUXTTEUCD.JRTCKXETXF': {'offerTermCod...
3,Z9UMVX66CBK745V7,{'Z9UMVX66CBK745V7.JRTCKXETXF': {'offerTermCod...
4,A66WQEVBSFUP6U3T,{'A66WQEVBSFUP6U3T.JRTCKXETXF': {'offerTermCod...


Θα δουλέψουμε αρχικά με το 2ο dataframe το οποίο περιέχει τα δεδομένα τιμολόγησης. Κρίνονται απαραίτητες 4 ενέργειες
- Άνοιγμα 2η στήλης και άπλωμα δεδομένων
- Αντιστοιχία εσωτερικού sku με αυτό που έβαλα εγώ, και πέταμα μίας στήλης εκ των 2
- Μελέτη για εντοπισμό καθολικών στηλών 
- Αφαίρεση περιττών στηλών
- Κατανόηση pricing και αντιστοίχιση με τους άλλους παρόχους